# Primary comparative — electronic component classification

**Self-contained notebook** — all code is in the cells below. Dependencies are managed with **`uv`** (see README setup).

Compares three families on Kaggle [Electronic Component Recognizer](https://www.kaggle.com/datasets/mdfaisalahmed025/electronic-component-recognizer) (~3.6k images, 20 classes):

1. **Classical** — HOG → Linear SVM  
2. **CNN from scratch** — SmallCNN  
3. **Transfer** — ResNet18  

Fairness: stratified 70/15/15 (frozen), accuracy + macro-F1 + confusion matrices, early stopping on val macro-F1.  
Figures show inline **and** save under `outputs/figures/` so shared notebooks keep plots.


## 0. Environment (`uv` only)

Prefer a project env from the repo (see **README → Setup**):

```bash
uv sync --all-groups
uv run python -m ipykernel install --user --name=compclass --display-name="Python (compclass)"
```

Select kernel **Python (compclass)** / `.venv/bin/python`. If that kernel already has the deps, you can skip the next cell.

The install cell uses the **`uv` CLI** (`uv pip install --python <this kernel>`), never bare `pip`.


In [ ]:
# Install into the *current kernel* with uv (skip if you already ran `uv sync`).
import shutil
import subprocess
import sys

if shutil.which("uv") is None:
    raise RuntimeError(
        "uv not found on PATH. Install it first:\n"
        "  curl -LsSf https://astral.sh/uv/install.sh | sh\n"
        "Docs: https://docs.astral.sh/uv/getting-started/installation/"
    )

pkgs = [
    "torch",
    "torchvision",
    "scikit-learn",
    "scikit-image",
    "pandas",
    "matplotlib",
    "seaborn",
    "pillow",
    "tqdm",
    "joblib",
    "kaggle",
]
cmd = ["uv", "pip", "install", "--python", sys.executable, *pkgs]
print("Running:", " ".join(cmd), flush=True)
subprocess.check_call(cmd)
print("Done. Restart the kernel if imports still fail.", flush=True)


## 1. Helpers + config


In [ ]:
# ===== Self-contained helpers (no external project package) =====
from __future__ import annotations

import json
import os
import random
import time
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from skimage.color import rgb2gray
from skimage.feature import hog
from skimage.transform import resize
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

# Paths: everything lands next to wherever you run the notebook
ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
SPLITS_DIR = DATA_DIR / "splits"
OUTPUTS_DIR = ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
WEIGHTS_DIR = OUTPUTS_DIR / "weights"
METRICS_DIR = OUTPUTS_DIR / "metrics"

KAGGLE_DATASET = "mdfaisalahmed025/electronic-component-recognizer"
DATASET_DIRNAME = "electronic-component-recognizer"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SEED = 42
IMAGE_SIZE = 224
SPLIT_RATIOS = (0.70, 0.15, 0.15)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def ensure_output_dirs() -> None:
    for path in (RAW_DIR, SPLITS_DIR, FIGURES_DIR, WEIGHTS_DIR, METRICS_DIR):
        path.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class Sample:
    path: str
    label: str


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _is_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def find_imagefolder_root(search_root: Path) -> Path | None:
    if not search_root.exists():
        return None
    candidates: list[Path] = []
    for directory in [search_root, *search_root.rglob("*")]:
        if not directory.is_dir():
            continue
        subdirs = [p for p in directory.iterdir() if p.is_dir() and not p.name.startswith(".")]
        if len(subdirs) < 2:
            continue
        class_like = 0
        for sub in subdirs:
            try:
                if any(_is_image(p) for p in sub.iterdir() if p.is_file()):
                    class_like += 1
            except PermissionError:
                continue
        if class_like >= 2 and class_like == len(subdirs):
            candidates.append(directory)
    if not candidates:
        return None
    candidates.sort(key=lambda p: (len(p.parts), -sum(1 for _ in p.iterdir() if _.is_dir())))
    return candidates[0]


def _extract_zips(raw_dir: Path) -> None:
    for zip_path in sorted(raw_dir.glob("*.zip")):
        extract_to = raw_dir / zip_path.stem
        if extract_to.exists() and find_imagefolder_root(extract_to):
            continue
        extract_to.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_to)


def _kaggle_auth_hint() -> str:
    return (
        "Kaggle authentication failed or is missing.\n"
        "Options:\n"
        "  1) `kaggle auth login` (OAuth)\n"
        "  2) Export KAGGLE_API_TOKEN from https://www.kaggle.com/settings/api\n"
        "  3) Save token to ~/.kaggle/access_token\n"
        "  4) Legacy: ~/.kaggle/kaggle.json\n"
        f"Or manual: drop the dataset zip/folder under {RAW_DIR}"
    )


def _download_kaggle(raw_dir: Path) -> Path:
    raw_dir.mkdir(parents=True, exist_ok=True)
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ImportError as exc:
        raise RuntimeError("Install kaggle or use the manual data drop.") from exc
    api = KaggleApi()
    try:
        api.authenticate()
    except Exception as exc:
        raise RuntimeError(_kaggle_auth_hint()) from exc
    print(f"Downloading {KAGGLE_DATASET} …")
    try:
        api.dataset_download_files(KAGGLE_DATASET, path=str(raw_dir), quiet=False)
    except Exception as exc:
        raise RuntimeError(f"Kaggle download failed.\n{_kaggle_auth_hint()}") from exc
    _extract_zips(raw_dir)
    root = find_imagefolder_root(raw_dir)
    if root is None:
        raise RuntimeError(f"Downloaded but no ImageFolder layout under {raw_dir}")
    return root


def resolve_dataset_root(*, prefer_kaggle: bool = True) -> Path:
    """Kaggle API if empty; else use zip/folder already in data/raw/."""
    ensure_output_dirs()
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    existing = find_imagefolder_root(RAW_DIR)
    if existing is not None:
        print(f"Using existing dataset at {existing}")
        return existing
    zips = list(RAW_DIR.glob("*.zip"))
    if zips:
        print(f"Found manual zip(s): {[z.name for z in zips]} — extracting …")
        _extract_zips(RAW_DIR)
        existing = find_imagefolder_root(RAW_DIR)
        if existing is not None:
            print(f"Using extracted dataset at {existing}")
            return existing
        raise RuntimeError(f"Zip extracted but no class folders found under {RAW_DIR}")
    if prefer_kaggle:
        return _download_kaggle(RAW_DIR)
    raise RuntimeError(
        f"No dataset found. Place a zip or ImageFolder under {RAW_DIR}, "
        "or enable Kaggle API credentials."
    )


def list_samples(dataset_root: Path) -> list[Sample]:
    samples: list[Sample] = []
    for class_dir in sorted(p for p in dataset_root.iterdir() if p.is_dir()):
        for image_path in sorted(class_dir.iterdir()):
            if _is_image(image_path):
                samples.append(Sample(path=str(image_path.resolve()), label=class_dir.name))
    if not samples:
        raise RuntimeError(f"No images found under {dataset_root}")
    return samples


def class_names_from_samples(samples: list[Sample]) -> list[str]:
    return sorted({s.label for s in samples})


def build_or_load_splits(
    samples: list[Sample],
    *,
    seed: int = SEED,
    ratios: tuple[float, float, float] = SPLIT_RATIOS,
    force: bool = False,
) -> dict[str, list[Sample]]:
    ensure_output_dirs()
    manifest_path = SPLITS_DIR / "split_manifest.json"
    meta_path = SPLITS_DIR / "split_meta.json"
    if manifest_path.exists() and not force:
        payload = json.loads(manifest_path.read_text())
        splits = {k: [Sample(**row) for row in rows] for k, rows in payload.items()}
        print(f"Loaded frozen split from {manifest_path}")
        return splits

    train_ratio, val_ratio, test_ratio = ratios
    paths = [s.path for s in samples]
    labels = [s.label for s in samples]
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        paths, labels, test_size=(1.0 - train_ratio), random_state=seed, stratify=labels
    )
    relative_val = val_ratio / (val_ratio + test_ratio)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=(1.0 - relative_val), random_state=seed, stratify=temp_labels
    )
    splits = {
        "train": [Sample(path=p, label=y) for p, y in zip(train_paths, train_labels, strict=True)],
        "val": [Sample(path=p, label=y) for p, y in zip(val_paths, val_labels, strict=True)],
        "test": [Sample(path=p, label=y) for p, y in zip(test_paths, test_labels, strict=True)],
    }
    manifest_path.write_text(
        json.dumps({k: [{"path": s.path, "label": s.label} for s in v] for k, v in splits.items()}, indent=2)
    )
    counts = {}
    for split_name, split_samples in splits.items():
        bag: dict[str, int] = defaultdict(int)
        for sample in split_samples:
            bag[sample.label] += 1
        counts[split_name] = dict(sorted(bag.items()))
    meta_path.write_text(
        json.dumps(
            {
                "seed": seed,
                "ratios": ratios,
                "n_train": len(splits["train"]),
                "n_val": len(splits["val"]),
                "n_test": len(splits["test"]),
                "class_counts": counts,
            },
            indent=2,
        )
    )
    print(f"Wrote frozen split to {manifest_path}")
    return splits


def summarize_splits(splits: dict[str, list[Sample]]) -> None:
    for name, items in splits.items():
        print(f"  {name:5s}: {len(items):4d} images")


def load_rgb(path: str | Path, size: int = IMAGE_SIZE) -> np.ndarray:
    image = Image.open(path).convert("RGB")
    array = np.asarray(image, dtype=np.float32) / 255.0
    return resize(array, (size, size), anti_aliasing=True, preserve_range=True).astype(np.float32)


def hog_vector(image_rgb: np.ndarray) -> np.ndarray:
    gray = rgb2gray(image_rgb)
    return hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    ).astype(np.float32)


def extract_hog_matrix(samples: list[Sample], *, size: int = IMAGE_SIZE, show_progress: bool = True):
    labels = sorted({s.label for s in samples})
    label_to_idx = {name: i for i, name in enumerate(labels)}
    features, targets = [], []
    iterator = tqdm(samples, desc="HOG") if show_progress else samples
    for sample in iterator:
        features.append(hog_vector(load_rgb(sample.path, size=size)))
        targets.append(label_to_idx[sample.label])
    return np.stack(features), np.asarray(targets, dtype=np.int64), labels


def train_hog_svm(train_samples, val_samples=None, *, C: float = 1.0, max_iter: int = 5000) -> dict:
    x_train, y_train, class_names = extract_hog_matrix(train_samples)
    pipeline = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("svm", LinearSVC(C=C, max_iter=max_iter, dual="auto", random_state=42)),
        ]
    )
    t0 = time.perf_counter()
    pipeline.fit(x_train, y_train)
    train_seconds = time.perf_counter() - t0
    result = {
        "model": pipeline,
        "class_names": class_names,
        "train_seconds": train_seconds,
        "n_features": int(x_train.shape[1]),
    }
    if val_samples is not None:
        x_val, y_val, _ = extract_hog_matrix(val_samples)
        y_hat = pipeline.predict(x_val)
        result["best_val_macro_f1"] = float(
            f1_score(y_val, y_hat, average="macro", labels=list(range(len(class_names))), zero_division=0)
        )
    coef = pipeline.named_steps["svm"].coef_
    intercept = pipeline.named_steps["svm"].intercept_
    result["params"] = int(coef.size + intercept.size)
    return result


def predict_hog_svm(pipeline, samples):
    x, y_true, class_names = extract_hog_matrix(samples)
    return y_true.tolist(), pipeline.predict(x).tolist(), class_names


class SmallCNN(nn.Module):
    def __init__(self, num_classes: int) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(256, 384, 3, padding=1), nn.BatchNorm2d(384), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(384, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.pool(self.features(x)))


def build_resnet18(num_classes: int, *, pretrained: bool = True) -> nn.Module:
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_transforms(train: bool, *, size: int = IMAGE_SIZE) -> transforms.Compose:
    if train:
        return transforms.Compose(
            [
                transforms.Resize((size, size)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(0.15, 0.15, 0.1),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ]
        )
    return transforms.Compose(
        [
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )


class ComponentDataset(Dataset):
    def __init__(self, samples, class_to_idx, transform=None):
        self.samples = samples
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(sample.path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, self.class_to_idx[sample.label]


def make_loaders(splits, class_names, *, batch_size=16, num_workers=0):
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    loaders = {}
    for split_name, items in splits.items():
        ds = ComponentDataset(items, class_to_idx, transform=build_transforms(train=(split_name == "train")))
        loaders[split_name] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=(split_name == "train"),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )
    return loaders


@torch.no_grad()
def predict_loader(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for inputs, targets in loader:
        preds = model(inputs.to(device)).argmax(dim=1).cpu().tolist()
        y_pred.extend(preds)
        y_true.extend(targets.tolist())
    return y_true, y_pred


def macro_f1_from_logits(model, loader, device, num_classes: int) -> float:
    y_true, y_pred = predict_loader(model, loader, device)
    return float(f1_score(y_true, y_pred, average="macro", labels=list(range(num_classes)), zero_division=0))


def train_torch_model(
    model,
    loaders,
    *,
    device,
    num_classes,
    epochs=25,
    lr=1e-3,
    weight_decay=1e-4,
    patience=5,
    freeze_backbone_epochs=0,
):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    def set_backbone_trainable(trainable: bool) -> None:
        for name, param in model.named_parameters():
            if name.startswith("fc") or name.startswith("classifier"):
                param.requires_grad = True
            else:
                param.requires_grad = trainable

    if freeze_backbone_epochs > 0:
        set_backbone_trainable(False)

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    history = []
    best_f1 = -1.0
    best_state = None
    epochs_without_improve = 0
    t0 = time.perf_counter()

    for epoch in range(1, epochs + 1):
        if freeze_backbone_epochs > 0 and epoch == freeze_backbone_epochs + 1:
            set_backbone_trainable(True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr * 0.1, weight_decay=weight_decay)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

        model.train()
        running_loss, n_batches = 0.0, 0
        for inputs, targets in tqdm(loaders["train"], desc=f"epoch {epoch}/{epochs}", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item())
            n_batches += 1

        train_loss = running_loss / max(n_batches, 1)
        val_f1 = macro_f1_from_logits(model, loaders["val"], device, num_classes)
        scheduler.step(val_f1)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_macro_f1": val_f1})
        print(f"epoch {epoch:02d}  loss={train_loss:.4f}  val_macro_f1={val_f1:.4f}")

        if val_f1 > best_f1 + 1e-4:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                print(f"Early stopping at epoch {epoch} (best val macro-F1={best_f1:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return {
        "model": model,
        "history": history,
        "best_val_macro_f1": best_f1,
        "train_seconds": time.perf_counter() - t0,
        "seed": SEED,
    }


def compute_classification_metrics(y_true, y_pred, class_names) -> dict[str, Any]:
    labels = list(range(len(class_names)))
    per_class_f1 = f1_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0)),
        "per_class_f1": {name: float(s) for name, s in zip(class_names, per_class_f1, strict=True)},
        "classification_report": classification_report(
            y_true, y_pred, labels=labels, target_names=class_names, output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def plot_confusion_matrix(y_true, y_pred, class_names, *, title, save_path=None, figsize=(12, 10)):
    ensure_output_dirs()
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig


def metrics_table(rows):
    frame = pd.DataFrame(rows)
    preferred = ["model", "family", "accuracy", "macro_f1", "weighted_f1", "params", "train_seconds"]
    cols = [c for c in preferred if c in frame.columns] + [c for c in frame.columns if c not in preferred]
    return frame[cols].sort_values("macro_f1", ascending=False).reset_index(drop=True)


ensure_output_dirs()
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0
BATCH_SIZE = 16 if DEVICE.type == "cuda" else 8
print(f"cwd={ROOT}")
print(f"device={DEVICE}  batch_size={BATCH_SIZE}  num_workers={NUM_WORKERS}")
print(f"data/raw → {RAW_DIR}")


## 2. Load data (Kaggle **or** manual)

- **Kaggle:** credentials via `kaggle auth login`, `KAGGLE_API_TOKEN`, or `~/.kaggle/access_token`
- **Manual:** put the `.zip` or an ImageFolder (`ClassName/*.jpg`) under `data/raw/` next to where you run this notebook


In [ ]:
dataset_root = resolve_dataset_root(prefer_kaggle=True)
samples = list_samples(dataset_root)
class_names = class_names_from_samples(samples)
print(f"root: {dataset_root}")
print(f"images: {len(samples)}  classes: {len(class_names)}")
print(class_names)

splits = build_or_load_splits(samples)  # force=True to rebuild
summarize_splits(splits)


## 3. Classical — HOG + Linear SVM


In [ ]:
classical = train_hog_svm(splits["train"], splits["val"], C=1.0)
joblib.dump(classical["model"], WEIGHTS_DIR / "hog_linear_svm.joblib")
print(
    f"train_s={classical['train_seconds']:.1f}  params≈{classical['params']}  "
    f"val_macro_f1={classical.get('best_val_macro_f1')}"
)

y_true_c, y_pred_c, _ = predict_hog_svm(classical["model"], splits["test"])
metrics_classical = compute_classification_metrics(y_true_c, y_pred_c, class_names)
print(f"test accuracy={metrics_classical['accuracy']:.4f}  macro_f1={metrics_classical['macro_f1']:.4f}")

fig_c = plot_confusion_matrix(
    y_true_c, y_pred_c, class_names,
    title="HOG + Linear SVM — test confusion matrix",
    save_path=FIGURES_DIR / "cm_hog_svm.png",
)
plt.show()


## 4. CNN from scratch — SmallCNN


In [ ]:
loaders = make_loaders(splits, class_names, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
cnn = SmallCNN(num_classes=len(class_names))
print(f"SmallCNN params: {count_parameters(cnn):,}")

cnn_result = train_torch_model(
    cnn, loaders, device=DEVICE, num_classes=len(class_names),
    epochs=30, lr=1e-3, patience=5,
)
torch.save(cnn_result["model"].state_dict(), WEIGHTS_DIR / "small_cnn.pt")

y_true_cnn, y_pred_cnn = predict_loader(cnn_result["model"], loaders["test"], DEVICE)
metrics_cnn = compute_classification_metrics(y_true_cnn, y_pred_cnn, class_names)
print(
    f"test accuracy={metrics_cnn['accuracy']:.4f}  macro_f1={metrics_cnn['macro_f1']:.4f}  "
    f"train_s={cnn_result['train_seconds']:.1f}"
)

fig_cnn = plot_confusion_matrix(
    y_true_cnn, y_pred_cnn, class_names,
    title="SmallCNN (from scratch) — test confusion matrix",
    save_path=FIGURES_DIR / "cm_small_cnn.png",
)
plt.show()


## 5. Transfer learning — ResNet18


In [ ]:
resnet = build_resnet18(num_classes=len(class_names), pretrained=True)
print(f"ResNet18 trainable params: {count_parameters(resnet):,}")

resnet_result = train_torch_model(
    resnet, loaders, device=DEVICE, num_classes=len(class_names),
    epochs=25, lr=1e-3, patience=5, freeze_backbone_epochs=3,
)
torch.save(resnet_result["model"].state_dict(), WEIGHTS_DIR / "resnet18.pt")

y_true_r, y_pred_r = predict_loader(resnet_result["model"], loaders["test"], DEVICE)
metrics_resnet = compute_classification_metrics(y_true_r, y_pred_r, class_names)
print(
    f"test accuracy={metrics_resnet['accuracy']:.4f}  macro_f1={metrics_resnet['macro_f1']:.4f}  "
    f"train_s={resnet_result['train_seconds']:.1f}"
)

fig_r = plot_confusion_matrix(
    y_true_r, y_pred_r, class_names,
    title="ResNet18 (transfer) — test confusion matrix",
    save_path=FIGURES_DIR / "cm_resnet18.png",
)
plt.show()


## 6. Comparative summary


In [ ]:
summary_rows = [
    {
        "model": "HOG + Linear SVM",
        "family": "classical",
        "accuracy": metrics_classical["accuracy"],
        "macro_f1": metrics_classical["macro_f1"],
        "weighted_f1": metrics_classical["weighted_f1"],
        "params": classical["params"],
        "train_seconds": classical["train_seconds"],
    },
    {
        "model": "SmallCNN",
        "family": "cnn_from_scratch",
        "accuracy": metrics_cnn["accuracy"],
        "macro_f1": metrics_cnn["macro_f1"],
        "weighted_f1": metrics_cnn["weighted_f1"],
        "params": count_parameters(cnn_result["model"]),
        "train_seconds": cnn_result["train_seconds"],
    },
    {
        "model": "ResNet18",
        "family": "transfer",
        "accuracy": metrics_resnet["accuracy"],
        "macro_f1": metrics_resnet["macro_f1"],
        "weighted_f1": metrics_resnet["weighted_f1"],
        "params": count_parameters(resnet_result["model"]),
        "train_seconds": resnet_result["train_seconds"],
    },
]

table = metrics_table(summary_rows)
display(table)

payload = {
    "dataset": str(dataset_root),
    "class_names": class_names,
    "device": str(DEVICE),
    "summary": summary_rows,
    "per_model": {
        "hog_svm": metrics_classical,
        "small_cnn": {**metrics_cnn, "history": cnn_result["history"]},
        "resnet18": {**metrics_resnet, "history": resnet_result["history"]},
    },
}
out = METRICS_DIR / "primary_metrics.json"
out.write_text(json.dumps(payload, indent=2))
print(f"wrote {out}")
print(f"figures: {FIGURES_DIR}")


### Notes for the later LaTeX write-up

- If **transfer ≫ from-scratch ≫ classical**, the set is small/noisy relative to ImageNet features.
- If **classical is competitive**, coarse texture/shape may already separate classes.
- Confused pairs in the matrices are usually the interesting story for the essay.
